# 2. Ingeniería de Características (Feature Engineering)

En la ingeniería de características tradicional, solíamos mapear manualmente las columnas categóricas en DataFrames de pandas (por ejemplo, convirtiendo `Sex` de 'male'/'female' a 0/1, o aplicando One-Hot encoding manualmente en archivos intermedios como `data/titanic_feature.csv`).

Sin embargo, este enfoque manual tiene desventajas graves en producción:
- Propensión a errores por inconsistencias de categorías en tiempo real.
- Dificultad para mantener scripts paralelos de preprocesamiento.
- Riesgo de fuga de información (data leakage) al escalar los datos antes del split.

### Enfoque Moderno: Pipelines Automatizados

En lugar de crear archivos de datos preprocesados manualmente, utilizaremos el **ColumnTransformer** y **Pipeline** de `scikit-learn`. Este enfoque tiene grandes ventajas:
1. **Automatización Completa**: Cuando el frontend envía los datos crudos en formato JSON (ej. `Sex='female'`, `Embarked='C'`), el preprocesador empaquetado realiza la codificación One-Hot y el escalamiento estándar automáticamente en memoria.
2. **Previene Fugas**: El escalamiento estándar (`StandardScaler`) calcula la media y la desviación estándar **solo en el conjunto de entrenamiento**, y las aplica al de prueba, garantizando una correcta evaluación.
3. **Código Limpio**: Evita tener que escribir funciones de mapeo personalizadas en la API de Flask, haciendo que `app.py` sea sumamente conciso.

In [1]:
# Demostración del Preprocesamiento del Pipeline
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Cargar los datos limpios
df = pd.read_csv('data/titanic_clean.csv')
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]

# Definir columnas
numerical_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_cols = ['Sex', 'Embarked']

# Crear el transformador de columnas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Ajustar y transformar una muestra para ver los resultados mapeados
X_transformed = preprocessor.fit_transform(X)
print("Datos originales (primer registro):")
print(X.iloc[0])
print("\nDatos transformados por el preprocesador (primer registro):\n", X_transformed[0])

Datos originales (primer registro):
Pclass         3
Sex         male
Age         22.0
SibSp          1
Parch          0
Fare        7.25
Embarked       S
Name: 0, dtype: object

Datos transformados por el preprocesador (primer registro):
 [ 0.82737724 -0.56573646  0.43279337 -0.47367361 -0.50244517  0.
  1.          0.          0.          1.        ]


In [2]:
df.to_csv('./data/titanic_procesado.csv', index=False)